# 🦾 Use PaliGemma with JSON adapter

Training Toolkit is a framework focused on training LLM adapters. However, it would be cruel of us to leave you without any inference code at all.

Assuming you already got you trained adapter for PaliGemma, here's a simple walkthough of using it for inference.

In [ ]:
from dotenv import load_dotenv
from pathlib import Path
import sys


sys.path.append(Path("..").resolve().as_posix())
_ = load_dotenv()

## 1. Load the model

- HF 🤗 PEFT is going to identify the base model, load it, load the adapter and attach the two together.
- We use `JSONTokenizer` to match the postprocessing step done by the stardard `image_json_preset` from the Training Toolkit.

In [ ]:
from peft import AutoPeftModelForCausalLM
from transformers import AutoProcessor

from training_toolkit.common.tokenization_utils.json import (
    JSONTokenizer,
)

CHECKPOINT_PATH = "paligemma/adapter/checkpoint"

model = AutoPeftModelForCausalLM.from_pretrained(CHECKPOINT_PATH)
processor = AutoProcessor.from_pretrained(CHECKPOINT_PATH)
json_tokenizer = JSONTokenizer(processor)

## 2. Load and preprocess the sample

In [ ]:
from PIL import Image


image = Image.open("assets/test_image.jpg")
prompt = "extract JSON"

inputs = processor(images=image, text=prompt, return_tensors="pt")

## 3. Generate the JSON

In [ ]:
# Generate text

generated_ids = model.generate(**inputs, max_new_tokens=1024, do_sample=True)

generated_text = processor.batch_decode(
    generated_ids,
    skip_special_tokens=True,
)[0]

generated_json = json_tokenizer.decode(generated_text)